In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, when, window, max, min, avg, sum, count, lit, last, lead, lag, to_timestamp, first, row_number
from pyspark.sql.window import Window

In [0]:
@dp.table

def bronze_aqi():
    env = spark.conf.get("pipelines.env", "dev")
    source_table = f"{env}_catalog.aqi_strm_{env}.aqi"
    
    return spark.readStream.table(source_table) \
        .withColumn("latitude", col("latitude").try_cast("double")) \
        .withColumn("longitude", col("longitude").try_cast("double")) \
        .withColumn("max_value", col("max_value").try_cast("double")) \
        .withColumn("min_value", col("min_value").try_cast("double")) \
        .withColumn("avg_value", col("avg_value").try_cast("double")) \
        .withColumn("last_update", to_timestamp("last_update", "yyyy-MM-dd HH:mm:ss"))

In [0]:
silver_rules = {
    "country": "country is not null",
    "state": "state is not null",
    "city": "city is not null",
    "pollutant_id": "pollutant_id is not null",
    "avg_pollutant_level": "avg_pollutant_level is not null"
}

In [0]:
@dp.table
@dp.expect_all_or_drop(silver_rules)

def silver_aqi():
    df = spark.readStream.table("bronze_aqi")
    agg_df = df.groupBy("country", "state", "city", "pollutant_id").agg(max("max_value").alias("max_pollutant_level"), min("min_value").alias("min_pollutant_level"), avg("avg_value").alias("avg_pollutant_level"), first("latitude").alias("latitude"), first("longitude").alias("longitude"), max("last_update").alias("last_update"))

    agg_df = agg_df.withColumn("aqi_quality_category", when(col("avg_pollutant_level") <= 50, "Good").
                                                        when(col("avg_pollutant_level") <= 100, "Satisfactory").
                                                        when(col("avg_pollutant_level") <= 200, "Moderate").
                                                        when(col("avg_pollutant_level") <= 300, "Poor").
                                                        otherwise("Severe"))
    agg_df = agg_df.withColumn("alert_level", when(col("avg_pollutant_level") >= 300, "Critical").
                                                when(col("avg_pollutant_level") >= 200, "High").
                                                when(col("avg_pollutant_level") >= 100, "Medium").
                                                otherwise("Normal"))
    
    return agg_df

In [0]:
# display(agg_df)

### State Wise Top Dominating Cities

In [0]:
@dp.materialized_view()

def gold_aqi_dominating_cities():
    agg_df = spark.read.table("silver_aqi")
    dom_df = agg_df
    dom_window = Window.partitionBy("State").orderBy(col("avg_pollutant_level").desc())
    dom_df = dom_df.withColumn("pollutant_dominant_rank", row_number().over(dom_window))
    dom_df = dom_df.filter(col("pollutant_dominant_rank") == 1)

    return dom_df

### Top 10 Most Polluted States

In [0]:
@dp.materialized_view()

def gold_aqi_most_polluted_states():
    agg_df = spark.read.table("silver_aqi")
    mp_df = agg_df
    mp_df = mp_df.groupBy("state").agg(max("avg_pollutant_level").alias("avg_aqi")).orderBy(col("avg_aqi").desc()).limit(10)
    
    return mp_df
    

### Top 10 Cities With Best AQI

In [0]:
@dp.materialized_view()

def gold_aqi_best_aqi():
    agg_df = spark.read.table("silver_aqi")
    baqi_df = agg_df
    baqi_df = baqi_df.groupBy("state", "city").agg(avg("avg_pollutant_level").alias("avg_aqi")).orderBy(col("avg_aqi").asc()).limit(10)
    
    return baqi_df

### Severe AQI States

In [0]:
@dp.materialized_view()

def gold_aqi_severe():
    agg_df = spark.read.table("silver_aqi")
    sev_df = agg_df
    sev_df = sev_df.filter(col("aqi_quality_category") == "Severe")
    
    return sev_df